# CS781 Capstone -- Phase 6: Abstractive Legal Summarization

Six experiments, each starting from one of **my own Phase 1 language models** (no other pretrained model, no external API).
All four models are decoder-only, so summarization is framed as language modelling:
`[document head+tail] + "Summary:" + [summary] + <EOS>`, trained with the loss on the summary tokens only,
and summaries are generated with a KV-cached greedy decoder with repetition control.

| # | Phase 1 model | Technique | Notes |
|---|---|---|---|
| 1 | GQA | **LoRMA** (compulsory) | own implementation from Phases 2-4 |
| 2 | GPT-2 | Houlsby-style adapters | hand-written bottleneck after every block |
| 3 | SwiGLU | LoRA | hand-written, on attention + MLP projections |
| 4 | Prefix-LM | Prefix Tuning | learned per-layer key/value prefix fed through `layer_past` (run causal) |
| 5 | GQA | Prompt Tuning | learned soft-prompt embeddings in front of the input |
| 6 | half-layer student of the best of 1-5 | **Knowledge distillation** (mandatory) | soft-logit + hard-label loss, teacher = best of 1-5 on dev |


## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil, re, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE  = "Phase_6_Summ/Test/test.json"

MODEL_PATHS = {
    "gpt2":     "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":   "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":      "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm": "/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 768, 12, 12, 4

# ---- input / output budget: document + separator + summary (+ soft tokens) must fit in 1024 ----
DOC_HEAD, DOC_TAIL = 448, 192            # document tokens kept: first 448 + last 192
HEAD_CHARS, TAIL_CHARS = 5000, 2500      # only this much raw text is tokenized (much faster than the full document)
MAX_SUM_TOKENS = 256                     # training summaries are cut to this many tokens (including EOS)
SEP_TEXT = "\n\nSummary:\n"
MAX_SOFT_TOKENS = 32                     # largest number of prompt/prefix positions a technique may use

# ---- data split (no public dev set: hold out part of the training data) ----
DEV_SIZE, DEV_EPOCH_DOCS, DEV_FINAL_DOCS = 300, 64, 150

# ---- training (shared) ----
SAMPLES_PER_EPOCH = 3000                 # random documents per epoch (lower it if Kaggle time runs short)
BATCH_SIZE, GRAD_ACCUM = 4, 4            # effective batch 16; 1024-token sequences are memory-heavy
WEIGHT_DECAY, WARMUP_RATIO = 0.01, 0.10
SELECT_BY = "rougeL"                     # "rougeL" (dev-subset generation) or "loss"
USE_BERTSCORE = True                     # needs `bert-score` + internet for its scoring model; falls back to NaN

# ---- decoding ----
DEFAULT_DECODE = dict(max_new=256, rep_penalty=1.1, no_repeat_ngram=3, min_new=40)
GEN_BATCH = 16

# switch experiments on/off (each can run in its own session)
RUN = {"e1": True, "e2": True, "e3": True, "e4": True, "e5": True, "e6": True}
TEACHER_STATE_PATH = None                # set to a saved best_model.pt if the teacher ran in another session
TEACHER_EXP = None                       # e.g. "e1".."e5"; None = pick the best finished experiment automatically

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase6"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "dev_gen_path": os.path.join(d, "dev_generations.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
    }

print("Configuration loaded.")

## 4. Install helpers (`regex` for the tokenizer, `bert-score` for the BERTScore metric)

In [ ]:
!pip -q install regex bert-score

## 5. Tokenizer (same shared tokenizer as Phases 1-4)

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer differs!"
print("All four checkpoints share the same tokenizer.")

EOS_ID = tokenizer.eos_token_id
PAD_ID = EOS_ID
SEP_IDS = tokenizer.encode(SEP_TEXT)
need = DOC_HEAD + DOC_TAIL + len(SEP_IDS) + MAX_SUM_TOKENS + MAX_SOFT_TOKENS
assert need <= MODEL_CONTEXT_LENGTH, f"budget {need} exceeds the {MODEL_CONTEXT_LENGTH}-token context"
print(f"token budget: doc {DOC_HEAD}+{DOC_TAIL}, separator {len(SEP_IDS)}, summary {MAX_SUM_TOKENS}, soft {MAX_SOFT_TOKENS} = {need} <= {MODEL_CONTEXT_LENGTH}")

## 6. Load Phase 6 data and hold out a dev split

In [ ]:
FIRST_DOC_KEYS = ["document", "text", "judgment"]
FIRST_SUM_KEYS = ["summary", "reference_summary", "abstract"]

def first_key(d, keys):
    for k in keys:
        if k in d:
            return k
    raise KeyError(f"none of {keys} found; available keys: {list(d.keys())}")

def to_records(raw, has_summary):
    """Accepts a list of dicts, a dict id -> dict, or a dict id -> text. Returns [{'id', 'document', ('summary')}]."""
    items = list(raw.items()) if isinstance(raw, dict) else [(None, v) for v in raw]
    recs = []
    for k, v in items:
        if isinstance(v, str):
            if has_summary:
                raise ValueError("training data must contain summaries")
            recs.append({"id": k, "document": v})
        else:
            rec = {"id": v.get("id", k), "document": v[first_key(v, FIRST_DOC_KEYS)]}
            if has_summary:
                rec["summary"] = v[first_key(v, FIRST_SUM_KEYS)]
            recs.append(rec)
    return recs

In [ ]:
from huggingface_hub import hf_hub_download

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

raw_train, raw_test = hf_json(TRAIN_FILE), hf_json(TEST_FILE)
print("raw types:", type(raw_train).__name__, type(raw_test).__name__)
all_train = to_records(raw_train, True)
test_records = to_records(raw_test, False)

DEV_SIZE = min(DEV_SIZE, max(50, len(all_train) // 10))
perm = np.random.RandomState(SEED).permutation(len(all_train))
dev_records = [all_train[i] for i in perm[:DEV_SIZE]]
train_records = [all_train[i] for i in perm[DEV_SIZE:]]
print(f"train {len(train_records)} | dev {len(dev_records)} | test {len(test_records)}")
assert len({str(r["id"]) for r in test_records}) == len(test_records), "duplicate test ids"
print("example keys:", list(all_train[0].keys()), "| id:", all_train[0]["id"])

In [ ]:
# length statistics -> sanity check for the token budget
sample = train_records[:300]
sum_tok = np.array([len(tokenizer.encode(r["summary"][:20000])) for r in sample])
doc_words = np.array([len(r["document"].split()) for r in sample])
sum_words = np.array([len(r["summary"].split()) for r in sample])
pc = lambda a: {p: int(np.percentile(a, p)) for p in (10, 50, 90, 99)}
print("document words  :", pc(doc_words))
print("summary words   :", pc(sum_words))
print("summary tokens  :", pc(sum_tok), f"| share cut at MAX_SUM_TOKENS={MAX_SUM_TOKENS}: {(sum_tok > MAX_SUM_TOKENS - 1).mean():.0%}")

## 7. Dataset and collator

`encode_doc` tokenizes only the first `HEAD_CHARS` and last `TAIL_CHARS` characters (tokenizing a whole judgment with the pure-Python BPE is slow),
then keeps the first `DOC_HEAD` and last `DOC_TAIL` tokens. Token ids are cached in the record, so every document is tokenized once.

In [ ]:
def encode_doc(text):
    text = text.strip()
    if len(text) > HEAD_CHARS + TAIL_CHARS:
        head_ids = tokenizer.encode(text[:HEAD_CHARS])
        tail_txt = text[-TAIL_CHARS:]
        cut = tail_txt.find(" ")
        tail_ids = tokenizer.encode(tail_txt[cut + 1:] if cut != -1 else tail_txt)
        return head_ids[:DOC_HEAD] + tail_ids[-DOC_TAIL:]
    ids = tokenizer.encode(text)
    if len(ids) > DOC_HEAD + DOC_TAIL:
        ids = ids[:DOC_HEAD] + ids[-DOC_TAIL:]
    return ids


def prepare(r):
    """Cache token ids in the record: prompt = document + separator, sum_ids = summary + EOS."""
    if "prompt_ids" not in r:
        r["prompt_ids"] = encode_doc(r["document"]) + SEP_IDS
        if "summary" in r:
            r["sum_ids"] = tokenizer.encode(r["summary"].strip()[:MAX_SUM_TOKENS * 6])[:MAX_SUM_TOKENS - 1] + [EOS_ID]
    return r


class SummDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        return prepare(self.records[i])


def summ_collator(batch):
    seqs = [r["prompt_ids"] + r["sum_ids"] for r in batch]
    L = max(len(s) for s in seqs)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, r in enumerate(batch):
        n_p, s = len(r["prompt_ids"]), seqs[i]
        ids[i, :len(s)] = torch.tensor(s)
        labels[i, n_p:len(s)] = torch.tensor(r["sum_ids"])      # loss only on the summary (+EOS)
    return {"input_ids": ids, "labels": labels}


_b = summ_collator([SummDataset(train_records)[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in _b.items()}, "| summary tokens in batch:", int((_b["labels"] != -100).sum()))

## 8. Backbone architectures (same four classes as Phases 2-4) and loader

In [ ]:
import math as _m

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,           "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,              "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_lm(name):
    """Phase 1 language model WITH its lm_head (generation needs it)."""
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    print(f"[{name}] load_state_dict:", model.load_state_dict(state))
    return model

for _name in BACKBONE_REGISTRY:
    _tmp = load_lm(_name)
    print(f"{_name}: {sum(p.numel() for p in _tmp.parameters())/1e6:.1f}M params")
    del _tmp

## 9. Summarization wrapper

One wrapper for every technique. It supports a KV cache for generation, a learned soft prompt (Prompt Tuning), and a learned per-layer key/value
prefix passed to the attention layers as `layer_past` (Prefix Tuning). Prefix key/values take no position embeddings; soft-prompt embeddings occupy
the first positions. The loss is computed only at summary positions, and `lm_head` is applied only there (memory).

In [ ]:
class SummLM(nn.Module):
    def __init__(self, lm, n_prompt=0, n_prefix=0):
        super().__init__()
        self.lm = lm
        self.n_prompt, self.n_prefix = n_prompt, n_prefix
        self.reserved = n_prompt + n_prefix            # positions/keys not available to real tokens
        self.prompt = None
        self.prefix_k = self.prefix_v = None
        if n_prompt > 0:                               # initialise the soft prompt from random token embeddings
            idx = torch.randint(0, lm.wte.weight.size(0) - 1, (n_prompt,))
            self.prompt = nn.Parameter(lm.wte.weight[idx].detach().clone())
        if n_prefix > 0:
            attn = lm.h[0].attn
            n_kv = getattr(attn, "n_kv_head", attn.n_head)
            shape = (len(lm.h), n_kv, n_prefix, attn.head_dim)
            self.prefix_k = nn.Parameter(torch.randn(shape) * 0.02)
            self.prefix_v = nn.Parameter(torch.randn(shape) * 0.02)

    def _prefix_past(self, B):
        dt = torch.float16 if (USE_AMP and torch.is_autocast_enabled()) else torch.float32
        return [(self.prefix_k[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt),
                 self.prefix_v[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt)) for i in range(self.prefix_k.size(0))]

    def forward(self, input_ids, labels=None, past=None, use_cache=False):
        B, T = input_ids.shape
        dev = input_ids.device
        if past is None and self.prefix_k is not None:
            past = self._prefix_past(B)
        past_len = past[0][0].size(-2) if past is not None else 0

        n_extra = 0
        if self.prompt is not None and past is None:   # prefill with a soft prompt in front of the tokens
            P = self.prompt.size(0)
            tok = self.lm.wte(input_ids) + self.lm.wpe(torch.arange(P, P + T, device=dev))
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1), tok], dim=1)
            n_extra = P
        else:
            start = past_len - self.n_prefix           # prefix key/values do not consume positions
            x = self.lm.wte(input_ids) + self.lm.wpe(torch.arange(start, start + T, device=dev))
        x = self.lm.drop(x)

        presents = []
        for i, block in enumerate(self.lm.h):
            x, pres = block(x, layer_past=None if past is None else past[i])
            presents.append(pres)
        x = self.lm.ln_f(x)
        if n_extra:
            x = x[:, n_extra:]

        if labels is not None:                          # training / dev loss: only summary positions
            tgt = labels[:, 1:]
            sel = tgt != -100
            logits_sel = self.lm.lm_head(x[:, :-1][sel])
            loss = F.cross_entropy(logits_sel.float(), tgt[sel])
            return {"loss": loss, "logits_sel": logits_sel, "targets_sel": tgt[sel]}
        logits = self.lm.lm_head(x[:, -1:, :])[:, -1, :]  # generation: next-token logits only
        return {"logits": logits, "past": presents if use_cache else None}

print("SummLM defined.")

## 10. Adaptation techniques (all hand-written)

In [ ]:
class LoRMALinear(nn.Module):
    """LoRMA: W' = (I + (alpha/r) B A) W0 (multiplicative low-rank update), same layer as in Phases 2-4."""
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        BA = self.B @ self.A
        M = self.eye + (self.alpha / self.r) * BA
        W_eff = M @ self.base.weight
        return F.linear(x, W_eff, self.base.bias)


def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model


class LoRALinear(nn.Module):
    """LoRA: W' = W0 + (alpha/r) B A (additive low-rank update)."""
    def __init__(self, base_linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base_linear.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


def apply_lora(model, target_names, r=16, alpha=32, dropout=0.05):
    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRALinear(child, r=r, alpha=alpha, dropout=dropout))
    return model


class AdapterBottleneck(nn.Module):
    def __init__(self, n_embd, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(n_embd, bottleneck)
        self.act = nn.GELU(approximate='tanh')
        self.up = nn.Linear(bottleneck, n_embd)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)   # identity at initialisation

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class AdapterBlockWrapper(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of each transformer block (simplified placement)."""
    def __init__(self, block, n_embd, bottleneck=64):
        super().__init__()
        self.block = block
        for p in self.block.parameters():
            p.requires_grad = False
        self.adapter = AdapterBottleneck(n_embd, bottleneck)

    def forward(self, x, layer_past=None, attn_mask=None):
        x, present = self.block(x, layer_past=layer_past, attn_mask=attn_mask)
        return self.adapter(x), present


def apply_adapters(model, n_embd, bottleneck=64):
    for i in range(len(model.h)):
        model.h[i] = AdapterBlockWrapper(model.h[i], n_embd, bottleneck)
    return model

print("LoRMA, LoRA and adapters defined. (Prefix / Prompt Tuning live inside SummLM.)")

## 11. Generation and text metrics

Generation is greedy with a KV cache, batched by exact prompt length (most prompts have the same length, so there is no padding to mask).
Controls: repetition penalty, no-repeat n-gram blocking, minimum and maximum length.
ROUGE-1/L are implemented here (lower-cased alphanumeric tokens, F1); they approximate the official scorer. BERTScore uses the `bert-score` package if available.

In [ ]:
def _toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def rouge1_f1(ref, hyp):
    r, h = Counter(_toks(ref)), Counter(_toks(hyp))
    ov = sum((r & h).values())
    if ov == 0:
        return 0.0
    p, rc = ov / sum(h.values()), ov / sum(r.values())
    return 2 * p * rc / (p + rc)

def lcs_len(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rougeL_f1(ref, hyp):
    a, b = _toks(ref), _toks(hyp)
    l = lcs_len(a, b)
    if l == 0:
        return 0.0
    p, rc = l / len(b), l / len(a)
    return 2 * p * rc / (p + rc)

def rouge_scores(refs, hyps):
    return {"rouge1": float(np.mean([rouge1_f1(r, h) for r, h in zip(refs, hyps)])),
            "rougeL": float(np.mean([rougeL_f1(r, h) for r, h in zip(refs, hyps)]))}

def repeated_trigram_rate(text):
    t = _toks(text)
    tri = list(zip(t, t[1:], t[2:]))
    return 1.0 - len(set(tri)) / len(tri) if tri else 0.0

_bert_scorer = None
def bertscore_f1(refs, hyps):
    """Mean BERTScore F1 (English default model). Returns NaN if the package or its model is unavailable."""
    global _bert_scorer
    if not USE_BERTSCORE:
        return float("nan")
    try:
        if _bert_scorer is None:
            from bert_score import BERTScorer
            _bert_scorer = BERTScorer(lang="en")
        _, _, f1 = _bert_scorer.score(hyps, refs)
        return float(f1.mean())
    except Exception as e:
        print("BERTScore unavailable:", repr(e)[:150])
        return float("nan")

def banned_next(ngram_map, toks, n):
    """Tokens that would repeat an n-gram already generated."""
    if n < 2 or len(toks) < n - 1:
        return ()
    return ngram_map.get(tuple(toks[-(n - 1):]), ())

def register_ngram(ngram_map, toks, n):
    if n >= 2 and len(toks) >= n:
        ngram_map.setdefault(tuple(toks[-n:-1]), set()).add(toks[-1])

In [ ]:
@torch.no_grad()
def generate_group(model, prompts, max_new=256, rep_penalty=1.1, no_repeat_ngram=3, min_new=40):
    """Greedy decoding for prompts of EQUAL length. Returns a list of generated id lists (EOS excluded)."""
    model.eval()
    B, plen = len(prompts), len(prompts[0])
    steps = min(max_new, MODEL_CONTEXT_LENGTH - model.reserved - plen)
    ids = torch.tensor(prompts, dtype=torch.long, device=device)
    out = [[] for _ in range(B)]
    done = [False] * B
    maps = [dict() for _ in range(B)]
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        res = model(ids, use_cache=True)
    past, logits = res["past"], res["logits"].float()
    for step in range(steps):
        logits = logits.clone()
        for b in range(B):
            if done[b]:
                continue
            toks = out[b]
            if rep_penalty != 1.0 and toks:
                idx = torch.tensor(sorted(set(toks)), device=device)
                l = logits[b, idx]
                logits[b, idx] = torch.where(l > 0, l / rep_penalty, l * rep_penalty)
            ban = banned_next(maps[b], toks, no_repeat_ngram)
            if ban:
                logits[b, list(ban)] = -float("inf")
            if step < min_new:
                logits[b, EOS_ID] = -float("inf")
        nxt = logits.argmax(-1)
        for b, t in enumerate(nxt.tolist()):
            if done[b]:
                continue
            if t == EOS_ID:
                done[b] = True
                continue
            out[b].append(t)
            register_ngram(maps[b], out[b], no_repeat_ngram)
        if all(done) or step == steps - 1:
            break
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            res = model(nxt.unsqueeze(1), past=past, use_cache=True)
        past, logits = res["past"], res["logits"].float()
    return out


def generate_summaries(model, records, cfg=None, batch_size=GEN_BATCH):
    cfg = {**DEFAULT_DECODE, **(cfg or {})}
    prompts = [prepare(r)["prompt_ids"] for r in records]
    by_len = {}
    for i, p in enumerate(prompts):
        by_len.setdefault(len(p), []).append(i)
    results = [""] * len(records)
    for L, idxs in sorted(by_len.items()):
        for s in range(0, len(idxs), batch_size):
            chunk = idxs[s:s + batch_size]
            outs = generate_group(model, [prompts[i] for i in chunk], **cfg)
            for i, o in zip(chunk, outs):
                results[i] = tokenizer.decode(o).strip()
    return results


def evaluate_dev(model, n_docs, cfg=None, with_bert=False):
    recs = dev_records[:n_docs]
    hyps = generate_summaries(model, recs, cfg)
    refs = [r["summary"] for r in recs]
    res = rouge_scores(refs, hyps)
    res["avg_gen_words"] = float(np.mean([len(h.split()) for h in hyps]))
    res["avg_ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    res["repeat_trigram_rate"] = float(np.mean([repeated_trigram_rate(h) for h in hyps]))
    if with_bert:
        res["bertscore_f1"] = bertscore_f1(refs, hyps)
    return res, hyps

print("Generation and metrics defined.")

## 12. Training, distillation loss, and the experiment runner

`run_experiment(spec)` builds the model for a technique, trains it (checkpoint chosen by dev ROUGE-L on a fixed dev subset), evaluates on the dev split
(ROUGE-1/L, BERTScore), generates the test summaries, and writes `prediction.json` (list format) + `submission.zip` + `run_info.json`.

In [ ]:
def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, path):
    state = torch.load(path, map_location="cpu")
    res = model.load_state_dict(state, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return model

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tr, sum(p.numel() for p in model.parameters())

def kd_loss(s_logits, t_logits, targets, T=2.0, alpha=0.5):
    """alpha * hard cross-entropy + (1-alpha) * T^2 * KL(teacher || student) on the summary positions."""
    s, t = s_logits.float(), t_logits.float()
    ce = F.cross_entropy(s, targets)
    kl = F.kl_div(F.log_softmax(s / T, dim=-1), F.softmax(t / T, dim=-1), reduction="batchmean") * (T * T)
    return alpha * ce + (1 - alpha) * kl

@torch.no_grad()
def eval_loss(model, records):
    loader = torch.utils.data.DataLoader(SummDataset(records), batch_size=4, shuffle=False, collate_fn=summ_collator)
    model.eval()
    tot, n = 0.0, 0
    for b in loader:
        ids, labels = b["input_ids"].to(device), b["labels"].to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(ids, labels=labels)
        k = (labels[:, 1:] != -100).sum().item()
        tot += out["loss"].item() * k; n += k
    return tot / max(n, 1)


def train_model(model, groups, spec, paths, teacher=None):
    epochs, patience = spec["epochs"], spec["patience"]
    train_ds = SummDataset(train_records)
    n_per_epoch = min(SAMPLES_PER_EPOCH, len(train_ds))
    steps_per_epoch = math.ceil(math.ceil(n_per_epoch / BATCH_SIZE) / GRAD_ACCUM)
    total_steps = steps_per_epoch * epochs
    warmup = max(1, int(WARMUP_RATIO * total_steps))
    optimizer = torch.optim.AdamW([{"params": p, "lr": lr} for p, lr in groups], weight_decay=WEIGHT_DECAY)
    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    params = [p for p in model.parameters() if p.requires_grad]
    kd_cfg = spec.get("kd", {})

    history, best, bad = [], None, 0
    t0 = time.time()
    for epoch in range(epochs):
        sampler = torch.utils.data.RandomSampler(train_ds, replacement=False, num_samples=n_per_epoch)
        loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=summ_collator)
        model.train()
        if teacher is not None:
            teacher.eval()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(loader):
            ids, labels = b["input_ids"].to(device), b["labels"].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(ids, labels=labels)
                loss = out["loss"]
                if teacher is not None:
                    with torch.no_grad():
                        t_out = teacher(ids, labels=labels)
                    loss = kd_loss(out["logits_sel"], t_out["logits_sel"], out["targets_sel"], **kd_cfg)
            scaler.scale(loss / GRAD_ACCUM).backward()
            run += loss.item(); cnt += 1
            if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            if (i + 1) % 100 == 0:
                print(f"  epoch {epoch+1} | batch {i+1}/{len(loader)} | running loss {run/cnt:.4f} | {time.time()-t0:.0f}s", flush=True)

        d_loss = eval_loss(model, dev_records[:DEV_EPOCH_DOCS])
        d_res, _ = evaluate_dev(model, DEV_EPOCH_DOCS)
        score = d_res["rougeL"] if SELECT_BY == "rougeL" else -d_loss
        history.append({"epoch": epoch + 1, "train_loss": run / cnt, "dev_loss": d_loss, **d_res})
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev loss {d_loss:.4f} | dev ROUGE-1 {d_res['rouge1']:.4f} "
              f"ROUGE-L {d_res['rougeL']:.4f} | gen words {d_res['avg_gen_words']:.0f} (ref {d_res['avg_ref_words']:.0f}) | {time.time()-t0:.0f}s", flush=True)
        if best is None or score > best:
            best, bad = score, 0
            torch.save(trainable_state(model), paths["model_path"])
        else:
            bad += 1
            if bad >= patience:
                print(f"early stopping at epoch {epoch+1}")
                break
    return history, time.time() - t0

In [ ]:
def build_model_for(spec):
    """Phase 1 backbone + one adaptation technique. Returns (model, [(params, lr)])."""
    lm = load_lm(spec["backbone"])
    for p in lm.parameters():
        p.requires_grad = False
    tech = spec["technique"]
    if tech == "lorma":
        model = SummLM(apply_lorma(lm, spec["targets"], r=spec["r"], alpha=spec["alpha"]))
    elif tech == "lora":
        model = SummLM(apply_lora(lm, spec["targets"], r=spec["r"], alpha=spec["alpha"]))
    elif tech == "adapters":
        model = SummLM(apply_adapters(lm, N_EMBD, bottleneck=spec["bottleneck"]))
    elif tech == "prefix":
        model = SummLM(lm, n_prefix=spec["n_soft"])
    elif tech == "prompt":
        model = SummLM(lm, n_prompt=spec["n_soft"])
    else:
        raise ValueError(tech)
    assert model.reserved <= MAX_SOFT_TOKENS
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    return model, [(params, spec["lr"])]


def build_student(spec):
    """Half-layer copy of the teacher's Phase 1 backbone, initialised from every other Phase 1 layer, fully trainable."""
    reg = BACKBONE_REGISTRY[spec["backbone"]]
    keep = list(range(0, N_LAYER, 2))                        # layers 0,2,4,...,10
    kwargs = dict(reg["kwargs"]); kwargs["n_layer"] = len(keep)
    lm = reg["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **kwargs)
    state = load_file(os.path.join(MODEL_PATHS[spec["backbone"]], "model.safetensors"))
    new = {}
    for k, v in state.items():
        if k.startswith("h."):
            parts = k.split(".")
            if int(parts[1]) in keep:
                parts[1] = str(keep.index(int(parts[1])))
                new[".".join(parts)] = v
        else:
            new[k] = v
    print("[student] load_state_dict:", lm.load_state_dict(new))
    model = SummLM(lm).to(device)
    for p in model.parameters():
        p.requires_grad = True
    return model, [([p for p in model.parameters()], spec["lr"])]


def pick_teacher():
    """Best finished experiment among e1..e5 by final dev ROUGE-L (or the one named in TEACHER_EXP)."""
    specs = {k: v for k, v in EXPERIMENT_SPECS.items() if k != "e6"}
    if TEACHER_EXP:
        return specs[TEACHER_EXP]
    best, best_spec = -1.0, None
    for k, s in specs.items():
        p = experiment_dirs(s["name"])["run_info_path"]
        if os.path.exists(p):
            sc = json.load(open(p)).get("dev", {}).get("rougeL", -1)
            if sc > best:
                best, best_spec = sc, s
    assert best_spec is not None, "no finished experiment found: run e1-e5 first or set TEACHER_EXP / TEACHER_STATE_PATH"
    print("teacher:", best_spec["name"], f"(dev ROUGE-L {best:.4f})")
    return best_spec


def run_experiment(spec):
    name = spec["name"]
    paths = experiment_dirs(name)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"===== {name} =====")
    teacher, teacher_name = None, None
    if spec["technique"] == "kd":
        t_spec = pick_teacher()
        teacher_name = t_spec["name"]
        teacher, _ = build_model_for(t_spec)
        load_trainable(teacher, TEACHER_STATE_PATH or experiment_dirs(t_spec["name"])["model_path"])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        spec = {**spec, "backbone": t_spec["backbone"]}
        model, groups = build_student(spec)
    else:
        model, groups = build_model_for(spec)
    trainable, total = count_params(model)
    print(f"trainable {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

    history, train_secs = train_model(model, groups, spec, paths, teacher)
    load_trainable(model, paths["model_path"])
    if teacher is not None:
        del teacher; gc.collect(); torch.cuda.empty_cache()

    dev_res, dev_hyps = evaluate_dev(model, DEV_FINAL_DOCS, with_bert=True)
    print("FINAL DEV:", {k: round(v, 4) for k, v in dev_res.items()})
    with open(paths["dev_gen_path"], "w", encoding="utf-8") as f:
        json.dump([{"id": r["id"], "reference": r["summary"], "generated": h}
                   for r, h in zip(dev_records[:20], dev_hyps[:20])], f, ensure_ascii=False, indent=1)

    t0 = time.time()
    test_hyps = generate_summaries(model, test_records)
    gen_secs = time.time() - t0
    write_submission(paths, test_hyps)

    info = {"experiment": name, "backbone": spec["backbone"], "technique": spec["technique"],
            "settings": {k: v for k, v in spec.items() if k not in ("name",)},
            "teacher": teacher_name, "trainable_params": trainable, "total_params": total,
            "batch": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "samples_per_epoch": SAMPLES_PER_EPOCH,
            "doc_head_tail": [DOC_HEAD, DOC_TAIL], "decode": DEFAULT_DECODE,
            "epochs_run": len(history), "history": history, "dev": dev_res,
            "train_seconds": round(train_secs), "test_gen_seconds_per_doc": round(gen_secs / max(1, len(test_records)), 3),
            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "checkpoint_mb": round(os.path.getsize(paths["model_path"]) / 1e6, 1)}
    with open(paths["run_info_path"], "w") as f:
        json.dump(info, f, indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()
    return info


def write_submission(paths, summaries):
    assert len(summaries) == len(test_records)
    sub = [{"id": r["id"], "summary": (s if s.strip() else "The court decided the appeal.")}
           for r, s in zip(test_records, summaries)]
    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    with zipfile.ZipFile(paths["submission_zip"]) as z:
        assert z.namelist() == ["prediction.json"]
    print("saved:", paths["submission_zip"], "| summaries:", len(sub))

print("Runner defined.")

## 13. The six experiments

Hyperparameters are starting points, not tuned values. Each experiment is one cell; set `RUN[...]` to `False` in the configuration to skip it.
Test the memory first with one batch if you change `BATCH_SIZE` (1024-token sequences are heavy).

In [ ]:
EXPERIMENT_SPECS = {
    "e1": dict(name="01_gqa_lorma", backbone="gqa", technique="lorma",
               targets=["q_proj", "k_proj", "v_proj", "c_proj"], r=8, alpha=16, lr=5e-4, epochs=6, patience=3),
    "e2": dict(name="02_gpt2_adapters", backbone="gpt2", technique="adapters",
               bottleneck=64, lr=1e-3, epochs=6, patience=3),
    "e3": dict(name="03_swiglu_lora", backbone="swiglu", technique="lora",
               targets=["c_attn", "c_proj", "w1", "w2", "w3"], r=16, alpha=32, lr=5e-4, epochs=6, patience=3),
    "e4": dict(name="04_prefixlm_prefix_tuning", backbone="prefixlm", technique="prefix",
               n_soft=32, lr=5e-3, epochs=6, patience=3),
    "e5": dict(name="05_gqa_prompt_tuning", backbone="gqa", technique="prompt",
               n_soft=32, lr=1e-2, epochs=6, patience=3),
    "e6": dict(name="06_kd_half_layer_student", backbone=None, technique="kd",
               lr=1e-4, epochs=6, patience=3, kd=dict(T=2.0, alpha=0.5)),
}
print({k: v["name"] for k, v in EXPERIMENT_SPECS.items()})

### Experiment 1 -- GQA + LoRMA (compulsory)

In [ ]:
if RUN["e1"]:
    info1 = run_experiment(EXPERIMENT_SPECS["e1"])

### Experiment 2 -- GPT-2 + Houlsby-style adapters

In [ ]:
if RUN["e2"]:
    info2 = run_experiment(EXPERIMENT_SPECS["e2"])

### Experiment 3 -- SwiGLU + LoRA

In [ ]:
if RUN["e3"]:
    info3 = run_experiment(EXPERIMENT_SPECS["e3"])

### Experiment 4 -- Prefix-LM + Prefix Tuning (causal)

In [ ]:
if RUN["e4"]:
    info4 = run_experiment(EXPERIMENT_SPECS["e4"])

### Experiment 5 -- GQA + Prompt Tuning

In [ ]:
if RUN["e5"]:
    info5 = run_experiment(EXPERIMENT_SPECS["e5"])

### Experiment 6 -- Knowledge distillation (mandatory)

Teacher = the best finished experiment among 1-5 (final dev ROUGE-L), rebuilt and loaded from its saved `best_model.pt`.
Student = the same architecture with 6 of the 12 layers, initialised from every other **Phase 1** layer (so it also starts from Phase 1),
trained with `0.5 * cross-entropy + 0.5 * T^2 * KL(teacher || student)` at temperature 2 on the summary positions.
The student has fewer layers but keeps the large embedding / output matrices, so the parameter saving is smaller than the layer saving; the report table gives the exact counts.

In [ ]:
if RUN["e6"]:
    info6 = run_experiment(EXPERIMENT_SPECS["e6"])

## 14. Decoding and length analysis (best experiment)

Compares decoding settings and generation lengths on the dev split with the best model from experiments 1-5, then optionally regenerates its test submission
with the best setting. `max_new` is limited by the 1024-token context minus the prompt.

In [ ]:
SWEEP = [
    dict(max_new=256, rep_penalty=1.0, no_repeat_ngram=0, min_new=40),   # plain greedy
    dict(max_new=256, rep_penalty=1.1, no_repeat_ngram=3, min_new=40),   # default
    dict(max_new=256, rep_penalty=1.2, no_repeat_ngram=4, min_new=40),
    dict(max_new=160, rep_penalty=1.1, no_repeat_ngram=3, min_new=40),   # shorter summaries
    dict(max_new=320, rep_penalty=1.1, no_repeat_ngram=3, min_new=80),   # longer summaries
]

def load_experiment_model(spec):
    model, _ = build_model_for(spec)
    load_trainable(model, experiment_dirs(spec["name"])["model_path"])
    return model

best_spec = pick_teacher()                     # best finished experiment among e1-e5
sweep_model = load_experiment_model(best_spec)
sweep_rows = []
for cfg in SWEEP:
    res, _ = evaluate_dev(sweep_model, DEV_FINAL_DOCS, cfg, with_bert=True)
    sweep_rows.append((cfg, res))
    print(cfg, "->", {k: round(v, 4) for k, v in res.items()})

In [ ]:
# optional: regenerate the test submission of the best experiment with the best sweep setting (ranked by BERTScore F1, ROUGE-L as fallback)
def _rank(r):
    b = r[1].get("bertscore_f1", float("nan"))
    return b if b == b else r[1]["rougeL"]
best_cfg = max(sweep_rows, key=_rank)[0]
print("best decoding setting:", best_cfg)
sweep_paths = experiment_dirs(best_spec["name"] + "_bestdecode")
write_submission(sweep_paths, generate_summaries(sweep_model, test_records, best_cfg))
del sweep_model; gc.collect(); torch.cuda.empty_cache()

## 15. Results table (all finished experiments)

In [ ]:
import glob
rows = []
for f in sorted(glob.glob(os.path.join(BASE_OUTPUT_DIR, "*", "run_info.json"))):
    ri = json.load(open(f))
    d = ri["dev"]
    rows.append((ri["experiment"], ri["technique"], ri["trainable_params"], ri["total_params"], d["rouge1"], d["rougeL"],
                 d.get("bertscore_f1", float("nan")), d["avg_gen_words"], d["repeat_trigram_rate"],
                 ri["train_seconds"], ri["test_gen_seconds_per_doc"], ri["peak_gpu_gb"], ri["checkpoint_mb"]))
print(f"{'experiment':28s} {'tech':9s} {'trainable':>11s} {'total':>11s} {'R1':>6s} {'RL':>6s} {'BERT':>6s} {'words':>6s} {'rep3':>6s} {'train_s':>8s} {'gen s/doc':>9s} {'GB':>5s} {'ckpt MB':>8s}")
for r in rows:
    print(f"{r[0]:28s} {r[1]:9s} {r[2]:11,d} {r[3]:11,d} {r[4]:6.3f} {r[5]:6.3f} {r[6]:6.3f} {r[7]:6.0f} {r[8]:6.3f} {r[9]:8d} {r[10]:9.3f} {r[11]:5.1f} {r[12]:8.1f}")